# RAD proposition-segments smoke test

This Colab notebook validates the feature branch, then compares a P1-style proposition run with the token-level segment-query run on the same small SkinCAP subset. Outputs remain under `/content`, not Drive.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/trisle0805/RAD-colab.git'
REPO_DIR = Path('/content/RAD-colab')
BRANCH = 'RAD-colab-segment-query'
SKINCAP_ROOT = Path('/content/drive/MyDrive/Cao học/Experiments/SkinCAP')
IMAGE_ROOT = SKINCAP_ROOT / 'skincap'
TRAIN_CSV = SKINCAP_ROOT / 'skincap_47_train.csv'
VAL_CSV = SKINCAP_ROOT / 'skincap_47_val.csv'
TEST_CSV = SKINCAP_ROOT / 'skincap_47_test.csv'
BERT_MODEL_ID = 'emilyalsentzer/Bio_ClinicalBERT'
BERT_DIR = Path('/content/models/Bio_ClinicalBERT')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)

COLAB_REQUIREMENTS = next(
    (path for path in (
        REPO_DIR / 'requirements/colab-skin.txt',
        REPO_DIR / 'requirements-colab-skin.txt',
    ) if path.is_file()),
    None,
)
assert COLAB_REQUIREMENTS is not None, 'Không tìm thấy file dependency Colab cho SkinCAP.'
subprocess.run(['pip', 'install', '-r', str(COLAB_REQUIREMENTS)], check=True)

if not (BERT_DIR / 'config.json').is_file():
    from huggingface_hub import snapshot_download

    BERT_DIR.parent.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=BERT_MODEL_ID, local_dir=str(BERT_DIR))

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for required in (TRAIN_CSV, VAL_CSV, TEST_CSV, IMAGE_ROOT, BERT_DIR):
    if not required.exists():
        raise FileNotFoundError(required)
print({
    'repo': str(REPO_DIR),
    'branch': BRANCH,
    'requirements': str(COLAB_REQUIREMENTS.relative_to(REPO_DIR)),
    'image_root': str(IMAGE_ROOT),
    'bert': str(BERT_DIR),
})

In [ ]:
# Cell 1: do not abbreviate pytest output; preserve all diagnostics.
pytest_result = subprocess.run(
    ['python', '-m', 'pytest', 'proposition/tests', '-q'],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(pytest_result.stdout)
if pytest_result.returncode:
    print(f'pytest failed with exit code {pytest_result.returncode}; inspect the complete output above.')
else:
    print('pytest passed.')

In [ ]:
# Cell 2: make deterministic CSV subsets, then run the two comparable smoke experiments.
import pandas as pd
import yaml

SMOKE_DATA = Path('/content/smoke_data')
SMOKE_RUNS = Path('/content/smoke_runs')
SMOKE_DATA.mkdir(exist_ok=True)
SMOKE_RUNS.mkdir(exist_ok=True)

subset_paths = {}
for source, row_count, name in ((TRAIN_CSV, 32, 'train'), (VAL_CSV, 16, 'val'), (TEST_CSV, 16, 'test')):
    frame = pd.read_csv(source).head(row_count)
    target = SMOKE_DATA / f'skincap_47_{name}.csv'
    frame.to_csv(target, index=False)
    subset_paths[name] = target
    print(name, frame.shape, target)

# Preserve the P2 batch/accumulation settings while limiting the smoke run to one epoch.
with (REPO_DIR / 'configs/skin.yaml').open(encoding='utf-8') as handle:
    smoke_config = yaml.safe_load(handle)
smoke_config['batch_size'] = 4
smoke_config['test_batch_size'] = 4
smoke_config['grad_accumulation_steps'] = 4
smoke_config['schedular']['epochs'] = 1
smoke_config['schedular']['warmup_epochs'] = 0
SMOKE_CONFIG = Path('/content/smoke_skin.yaml')
with SMOKE_CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(smoke_config, handle, sort_keys=False)

KB_PATH = REPO_DIR / 'kb_builder/outputs/final/kb_propositions.json'
assert KB_PATH.is_file(), f'Không tìm thấy KB: {KB_PATH}'
common = [
    'python', 'main_proposition.py',
    '--config', str(SMOKE_CONFIG),
    '--bert_model_name', str(BERT_DIR),
    '--kb_path', str(KB_PATH),
    '--max_length', '512',
    '--embed_dim', '768',
    '--topk', '1', '2', '3',
    '--train_csv', str(subset_paths['train']),
    '--val_csv', str(subset_paths['val']),
    '--test_csv', str(subset_paths['test']),
    '--image_root', str(IMAGE_ROOT),
    '--device', 'cuda', '--seed', '42',
]

def run_logged(command):
    completed = subprocess.run(command, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    return completed

p1_dir = SMOKE_RUNS / 'proposition'
p1 = run_logged(common + ['--output_dir', str(p1_dir), '--run_id', 'SMOKE_P1', '--query_level', 'proposition'])
if p1.returncode:
    raise RuntimeError(f'Proposition smoke run failed: {p1.returncode}')

segment_dir = SMOKE_RUNS / 'proposition_segments'
segment = run_logged(common + [
    '--output_dir', str(segment_dir), '--run_id', 'SMOKE_SEG',
    '--query_level', 'proposition_segments', '--query_chunk_size', '2048',
])
if segment.returncode:
    text = segment.stdout.lower()
    if 'out of memory' in text or 'cuda oom' in text:
        print('Segment smoke test ran out of GPU memory. Reduce --query_chunk_size and retry; do not change batch size, gradient accumulation, or epoch count.')
    else:
        raise RuntimeError(f'Segment smoke run failed: {segment.returncode}')

In [ ]:
# Cell 3: compare runtime/memory/artifacts and prove the segment analysis script runs.
import numpy as np

rows = []
for name, directory in [('proposition', p1_dir), ('proposition_segments', segment_dir)]:
    config_path = directory / 'run_config.json'
    if not config_path.exists():
        rows.append({'run': name, 'status': 'not available'})
        continue
    config = json.loads(config_path.read_text(encoding='utf-8'))
    runtime = config.get('runtime', {})
    rows.append({
        'run': name,
        'mean_epoch_seconds': runtime.get('mean_epoch_seconds'),
        'peak_gpu_memory_bytes': runtime.get('peak_gpu_memory_bytes'),
        'segment_query_stats': (
            json.loads((directory / 'segment_query_stats.json').read_text(encoding='utf-8'))
            if (directory / 'segment_query_stats.json').exists() else None
        ),
    })
display(pd.DataFrame(rows))

evidence = segment_dir / 'checklist' / 'test_segment_evidence.npz'
if evidence.exists():
    with np.load(evidence, allow_pickle=False) as artifact:
        evidence_shapes = {name: list(artifact[name].shape) for name in artifact.files}
    print(json.dumps(evidence_shapes, indent=2))
    analysis = subprocess.run(
        ['python', 'proposition/visualization/segment_map_analysis.py', '--evidence', str(evidence)],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(analysis.stdout)
    if analysis.returncode:
        raise RuntimeError(f'segment_map_analysis.py failed: {analysis.returncode}')
else:
    print('Segment evidence is unavailable because the segment smoke run did not complete.')